## Evaluation Metrics

We evaluate whether the LLM correctly selects `query_disasters` and extracts the right arguments from natural language queries.

We are going to use five eight questions, and we expect the LLM select the tool query_disaster and the right arguments based on the documentation. 

In [2]:
%pip install pandas mcp[cli] python-dotenv openai langchain langchain-openai -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
EVAL_DATASET = [
    {"query": "What earthquakes happened in Japan in 2011?",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "Japan", "year": "2011", "disaster_type": "Earthquake"}},
    {"query": "Show me floods in Colombia",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "Colombia", "disaster_type": "Flood"}},
    {"query": "Hurricanes in the United States in 2005",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "United States", "year": "2005", "disaster_type": "Hurricane"}},
    {"query": "How many people died in earthquakes in Chile?",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "Chile", "disaster_type": "Earthquake"}},
    {"query": "Natural disasters in India in 2020",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "India", "year": "2020"}},
    {"query": "Volcanic eruptions in Indonesia",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "Indonesia", "disaster_type": "Volcanic activity"}},
    {"query": "Droughts in Africa",
     "expected_tool": "query_disasters",
     "expected_args": {"disaster_type": "Drought"}},
    {"query": "Tsunamis in 2004",
     "expected_tool": "query_disasters",
     "expected_args": {"year": "2004", "disaster_type": "Earthquake"}},
]

## Set-up the LLM and dataset execution 

In [ ]:
from dotenv import load_dotenv
from openai import AzureOpenAI
import os
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("MODEL")
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT")

client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    api_version="2024-08-01-preview",
    azure_endpoint=AZURE_ENDPOINT,
)


def llm_client(message: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": message},
        ],
    )
    return response.choices[0].message.content


def get_prompt_to_identify_tool_and_arguments(query, tools):
    tools_description = "\n".join(
        [f"- {tool['name']}: {tool['description']} | args: {tool['args']}" for tool in tools]
    )
    return (
        "You are a helpful assistant with access to these tools:\n\n"
        f"{tools_description}\n"
        "Choose the appropriate tool based on the user's question.\n"
        f"User's Question: {query}\n"
        "If no tool is needed, reply directly.\n\n"
        "IMPORTANT: When you need to use a tool, you must ONLY respond with "
        "the exact JSON object format below, nothing else:\n"
        "{\n"
        '    "tool": "tool-name",\n'
        '    "arguments": {\n'
        '        "argument-name": "value"\n'
        "    }\n"
        "}\n\n"
    )


TOOLS = [
    {
        "name": "query_disasters",
        "description": """
            Query the natural disasters CSV dataset.

            Args:
                    country: Filter by country name (case-insensitive). E.g. "Argentina", "Australia". If None, no country filter is applied.
                    year: Filter by year (e.g. 1970). If None, no year filter is applied.
                    disaster_type: Filter by disaster type (case-insensitive). Supported types: "Animal accident", "Drought", "Earthquake", "Epidemic", "Extreme temperature", "Flood", "Fog", "Glacial lake outburst", "Impact", "Insect infestation", "Landslide", "Mass movement (dry)", "Storm", "Volcanic activity", "Wildfire". If None, no disaster type filter is applied.
                    limit: Maximum number of results to return

                Expected output: A JSON string containing a list of disasters matching the criteria, with key details for each disaster. If no disasters match, a message indicating no results found. If the dataset is not loaded, an error message is returned.
                {   "total": 3,
                    "disasters": [
                        {
                        "Year": 2021,
                        "Seq": 182,
                        "Disaster Group": "Natural",
                        "Disaster Subgroup": "Hydrological",
                        "Disaster Type": "Flood",
                        "Country": "Colombia",
                        "ISO": "COL",
                        "Region": "South America",
                        "Continent": "Americas",
                        "Location": "Florencia City (Caquetá Department); Quípama Town (Boyacá Department), Bogotá",
                        "Origin": "Heavy rains",
                        "Associated Dis": "Slide (land, mud, snow, rock)",
                        "Dis Mag Scale": "Km2",
                        "Start Year": 2021,
                        "Start Month": 4.0,
                        "Start Day": 1.0,
                        "End Year": 2021,
                        "End Month": 4.0,
                        "End Day": 5.0,
                        "Total Deaths": 3.0,
                        "No Injured": 5.0,
                        "No Affected": 360.0,
                        "Total Affected": 365.0,
                        "Adm Level": "2",
                        "Admin2 Code": "13608;13691;13914",
                        "Geo Locations": "Florencia, Quipama, Santafe De Bogota D.c. (Adm2). "
                        }
                    ]
                }        
""",
        "args": {
            "country": "str (optional) - filter by country name",
            "year": "int (optional) - filter by year",
            "disaster_type": "str (optional) - filter by disaster type (Flood, Storm, Earthquake, etc.)",
            "limit": "int (default 10) - max results",
        },
    }
]

correct_tool = 0
total_args = 0
correct_args = 0

print("Running evaluation...\n")
for i, entry in enumerate(EVAL_DATASET):
    query = entry["query"]
    prompt = get_prompt_to_identify_tool_and_arguments(query, TOOLS)
    response = llm_client(prompt)

    print(f"[{i+1}/{len(EVAL_DATASET)}] Query: {query}")
    print(f"  LLM response: {response[:200]}")

    try:
        parsed = json.loads(response)
        tool_match = parsed.get("tool") == entry["expected_tool"]
        if tool_match:
            correct_tool += 1
        print(f"  Tool selected: {parsed.get('tool')} {'[correct]' if tool_match else '[WRONG]'}")

        for key, expected_val in entry["expected_args"].items():
            total_args += 1
            actual_val = str(parsed.get("arguments", {}).get(key, ""))
            match = expected_val.lower() in actual_val.lower()
            if match:
                correct_args += 1
            print(f"  Arg '{key}': expected='{expected_val}' got='{actual_val}' {'[correct]' if match else '[WRONG]'}")
    except json.JSONDecodeError:
        print(f"  ERROR: Could not parse JSON from LLM response")

    print()

tool_accuracy = correct_tool / len(EVAL_DATASET) * 100
arg_accuracy = correct_args / total_args * 100 if total_args > 0 else 0

print(f"{'='*50}")
print(f"Tool Selection Accuracy: {correct_tool}/{len(EVAL_DATASET)} ({tool_accuracy:.1f}%)")
print(f"Argument Match Rate:     {correct_args}/{total_args} ({arg_accuracy:.1f}%)")
print(f"{'='*50}")

Running evaluation...

